# Clase 9: Astropy: Unidades y Coordenadas


Adaptado de http://www.astropy.org/astropy-tutorials/Coordinates.html

## **Introducción a Astropy**

* Librería de código abierto impulsado por la comunidad con herramientas diseñadas para investigación astronómica.
* Históricamente, el código astronómico era fragmentado, con lectores de FITS y conversores de unidades propios, lo que causaba errores y falta de reproducibilidad.
* Astropy establece un estándar unificado que se integra con el ecosistema científico de Python (`NumPy`, `SciPy`, `Matplotlib`).
* Instalado por defecto en el entorno de Colab.
* Desarrollado por Thomas Robitaille et al en el STScI en el 2011.
* Citación original: *Robitaille, Thomas P.; Tollerud, Erik J.; et al. (2013). "Astropy: A community Python package for astronomy". Astronomy & Astrophysics. 558: A33. arXiv:[1307.6212](https://arxiv.org/abs/1307.6212)*
* **Herramientas para la Minería de Datos:**
    * **Constants:** Valores físicos estandarizados.
    * **Units:** Gestión, conversión de unidades físicas.
    * **Tables:** Manejo de catálogos con metadatos y unidades.
    * **WCS (World Coordinate System):** Estándar (contenido en el *header* de los archivos FITS) que define la transformación matemática entre las coordenadas de píxeles $(x, y)$ y las coordenadas astronómicas $(RA, Dec)$.
    * Compatibilidad con `astroquery`, librería de acceso a múltiples bases de datos.

## **Constantes Físicas (`astropy.constants`)**

* **Valores CODATA:** Basados en estándares internacionales.
* **Metadatos:** Incluyen nombre, valor, incertidumbre y referencia.
* **Objetos `Quantity`:** Las constantes ya vienen con sus unidades asociadas.

**Ejemplo de uso:**
$$R_s = \frac{2GM}{c^2}$$
```python
from astropy.constants import G, M_sun, c

R_s = (2 * G * M_sun) / c**2
print(R_s.to('km'))
```

```python
import astropy.constants as c
# Es preferible usar un alias para legibilidad del código

R_s = (2 * c.G * c.M_sun) / c.c**2
print(R_s.to('km'))

## **Unidades y Cantidades (`astropy.units`)**

* El tipo de objeto `Quantity` es la unión entre un valor (o array de `NumPy`) con un objeto de unidad (dimensión) física.
* Permite hacer conversiones y simplificaciones sin errores manuales.
* También permite conversiones entre unidades que no son dimensionalmente iguales pero tienen una relación física:
    * **Espectrales:** Energía $\leftrightarrow$ Frecuencia $\leftrightarrow$ Longitud de onda.
    * **Astrofísicas:** Paralaje $\leftrightarrow$ Distancia.

**Ejemplo de código:**
```python
import astropy.units as u

distancia = 10 * u.pc
# Conversión simple
print(distancia.to(u.lightyear))

# Equivalencia de paralaje (arcsec -> pc)
p = 0.002 * u.arcsec
d = p.to(u.pc, equivalencies=u.parallax())

## **Tablas de Astropy (`astropy.table`)**

*   Objetos diseñados para leer/escribir archivos **FITS**, **VOTables** y **CSV** conservando metadatos.
*  Una sola celda puede contener un array (ej. un espectro completo de 2000 canales), permitiendo manejar estructuras de datos embebidos que el CSV estándar no soporta.
*   **`QTable` (Quantity Table):**
    *   Clase nativa de Astropy para tablas.
    *   Cada columna mantiene sus unidades y se comporta nativamente como un array de `astropy.units`.
*   **Operaciones de Minería:**
    *   Slicing mediante máscaras booleanas.
    *   Joins de catálogos (p. ej. cruzar datos de posición de Gaia con fotometría de SDSS).
*   **Compatibilidad**
    * Convierte archivos **FITS**, **VOTables**, **HDF**, a objetos `Qtables`.
    * Interopera tablas con  dataframes de `pandas`, diccionarios de Python, o arreglos de `numpy`. Exportación a Pandas DataFrame con el método `.to_pandas()`.
    * Los resultados de búsquedas TAP/ADQL en bases de datos como Gaia o SDSS se descargan como objetos que se convierten directamente en `QTable`.

## **Coordenadas Celestes (`astropy.coordinates`)**

*   Objetos `SkyCoord`, que unifican posiciones, sistemas de referencia y distancias.
*   Sistemas de referencia/coordenados: ICRS (estándar), FK5, Galáctico, AltAz (local), entre otros.
*   Convierte entre coordenadas ecuatoriales (RA/Dec), alt-az o galácticas ($l, b$).
* Permite calcular la distancia angular en el cielo entre dos objetos.
*   **Cross-matching:** Encontrar qué objetos de un catálogo A coinciden con el catálogo B dentro de un radio determinado (ej. 1 arcsec).
* Permite calcular la masa de aire para observaciones.

```python
from astropy.coordinates import SkyCoord
import astropy.units as u

# Definir una posición (ej. el centro de la Galaxia)
c = SkyCoord(ra=266.4168*u.deg, dec=-29.0078*u.deg, frame='icrs')

# Transformar a coordenadas galácticas
gal = c.galactic
print(f"Longitud: {gal.l:.2f}, Latitud: {gal.b:.2f}")

# Calcular distancia a otro objeto
otro = SkyCoord(ra=266.5*u.deg, dec=-28.9*u.deg, frame='icrs')
sep = c.separation(otro)
print(f"Separación: {sep.arcmin:.2f} arcmin")

## Unidades

Astropy puede hacer conversión de unidades muy eficientemente a través del método `.to()` aplicado a un objeto `Quantity`. También hay atajos como los métodos `.cgs` `.si`.

In [1]:
import astropy.units as u
measurement = 10*u.km/u.h
measurement.to(u.m/u.s)

<Quantity 2.77777778 m / s>

In [2]:
flux = 10*u.Jy
flux.cgs

<Quantity 1.e-22 g / s2>

Astropy incluye equivalencias para casi cualquier situación donde dos unidades no son físicamente la misma dimensión, pero están vinculadas por una ley física o un modelo estándar. Se usan con el alias, o sea `u.spectral()` como argumento del método de conversión `.to()`

| Equivalencia | Convierte entre... | Constante/Parámetro necesario |
| :--- | :--- | :--- |
| `spectral()` | $\lambda, \nu, E, cm^{-1}$ | Ninguno (usa $h$ y $c$) |
| `parallax()` | Ángulo ($arcsec$) y Distancia ($pc$) | Ninguno (geometría) |
| `doppler_x()` | Velocidad y $\lambda$ o $\nu$ | $\lambda_0$ o $\nu_0$ (reposo) |
| `mass_energy()` | Masa y Energía | Ninguno (usa $c^2$) |
| `spectral_density()`| $Jy$ y $erg \cdot s^{-1} \cdot cm^{-2} \cdot \text{\AA}^{-1}$ | $\lambda$ o $\nu$ de observación |

In [3]:
import astropy.units as u

lambda_uv = 91.2 * u.nm

# Con equivalencies=u.spectral() Astropy usa E = h * c / lambda
frecuencia = lambda_uv.to(u.GHz, equivalencies=u.spectral())
energia = lambda_uv.to(u.eV, equivalencies=u.spectral())
numero_de_onda = lambda_uv.to(u.cm**-1, equivalencies=u.spectral())

print(f"Longitud de onda: {lambda_verde}")
print(f"Frecuencia: {frecuencia:.2f}")
print(f"Energía: {energia:.2e}")
print(f"Número de onda (lambda^-1): {numero_de_onda:.2f}")


NameError: name 'lambda_verde' is not defined

Astropy puede descomponer unidades, reconocer unidades en texto plano, extraer valores, unidades de objetos `Quantity` o representar unidades en strings para latex. También puede revisar equivalencia dimensional.

In [ ]:
un_watt = 1 * u.W
un_watt.decompose()

In [ ]:
3*u.Unit("kg m2 / s2")

In [ ]:
distancia = 100 * u.lightyear
print(distancia.value) # 100
print(distancia.unit)  # ly

In [ ]:
u.Unit("kg m2 / s2").to_string('latex')

In [ ]:
freq = 5 * u.erg / (u.cm**2 * u.s * u.nm)
(freq.unit).is_equivalent(u.Jy)                # False

## Usando `astropy.coordinates`

En esta parte exploraremos cómo el paquete `astropy.coordinates` junto con la funcionalidad general de astropy puede ser usado para planear observaciones y en general obtener información que requiera el uso de catálogos astronómicos grandes.

In [ ]:
import urllib
import IPython.display
import numpy as np

In [ ]:
%matplotlib inline
from matplotlib import pyplot as plt

In [ ]:
from astropy import units as u
from astropy.coordinates import SkyCoord

## Determinando lugares en el cielo con `coordinates`

Comencemos considerando un campo alrededor del pintoresco Grupo Compacto Hickson 7 (HCG7). Primero necesitamos obtener un objeto que represente las coordenadas del centro de este grupo.

En Astropy, el objeto más común para trabajar coordenadas es `SkyCoord`.  Un `SkyCoord` puede ser creado simplemente a partir de ángulos como se muestra más abajo. Es recomendable especificar el marco de referencia de sus coordenadas, aunque no es estrictamente necesario, ya que por defecto se usa el estándar ICRS de la IAU. Este estándar se puede entender como aproximadamente igual al sistema ecuatorial en el equinoxio de J2000, y se da en RA/DEC.

https://en.wikipedia.org/wiki/International_Celestial_Reference_System

In [ ]:
SkyCoord?

In [ ]:
hcg7_center = SkyCoord(9.81625*u.deg, 0.88806*u.deg, frame='icrs')
hcg7_center

`SkyCoord` también acepta coordenadas formateadas como `strings` ya sea como  `strings` separados para RA/DEC o como un `string` simple. Si las unidades no vienen en el `string` original, usted tiene que especificarlas.

In [ ]:
SkyCoord('23h39m15.9s', '0d53m17.016s', frame='icrs')

In [ ]:
SkyCoord('0:39:15.9 0:53:17.016', unit=(u.hour, u.deg), frame='icrs')

Si el objeto que le interesa está en [SESAME](http://cdsweb.u-strasbg.fr/cgi-bin/Sesame), usted puede hallarlo directamente usando el método de clase<sup>1</sup> `SkyCoord.from_name()`. Esto requiere que usted esté conectado a internet. Dado que ya lo definimos explícitamente, no es necesario ejecutar este paso si usted no está conectado a internet.

<sub> <sup>1</sup>Un objeto de clase es un constructor alternativo para un objeto `SkyCoord` object. Si usted llama la  `SkyCoord.from_name()` con un nombre, va a generar un nuevo objeto `SkyCoord`. Para más información sobre los objetos de clase y su uso, vea [esta página](https://julien.danjou.info/blog/2013/guide-python-static-class-abstract-methods).</sub>

In [ ]:
hcg7_center = SkyCoord.from_name('Hcg 7')
hcg7_center

In [ ]:
Black_Hole = SkyCoord.from_name('Sgr A*')
Black_Hole

El objeto que acabamos de crear nos da varias formas útiles de tener acceso a la información que almacena. En particular se usan los atributos ``ra`` y ``dec``, que son objetos [``Quantity``](http://docs.astropy.org/en/stable/units/index.html) de Astropy especializados (de hecho, pertenecen a una sub-clase llamada [``Angle``](http://docs.astropy.org/en/stable/api/astropy.coordinates.Angle.html), cuyas subclases son [``Latitude``](http://docs.astropy.org/en/stable/api/astropy.coordinates.Latitude.html) y [``Longitude``](http://docs.astropy.org/en/stable/api/astropy.coordinates.Longitude.html)).  Estos objetos almacenan ángulos y muestran representaciones bonitas de ellos. También tiene atributos útiles para convertir fácilmente a unidades comunes de ángulo.

In [ ]:
type(hcg7_center.ra), type(hcg7_center.dec)

In [ ]:
hcg7_center.dec

In [ ]:
hcg7_center.ra

In [ ]:
hcg7_center.dec.hour

Ahora que tenemos un objeto `SkyCoord`, podemos usarlo para acceder a datos del [Sloan Digital Sky Survey](http://www.sdss.org/) (SDSS).  Empecemos intentando obtener una imagen usando la herramienta de recorte de imágenes de SDSS para asegurarnos que HGC7 está incluido en el catálogo SDSS y que tiene buena calidad de imagen.

In [ ]:
impix = 1024
imsize = 12*u.arcmin
cutoutbaseurl = 'http://skyservice.pha.jhu.edu/DR12/ImgCutout/getjpeg.aspx'
query_string = urllib.parse.urlencode(dict(ra=hcg7_center.ra.deg,
                                     dec=hcg7_center.dec.deg,
                                     width=impix, height=impix,
                                     scale=imsize.to(u.arcsec).value/impix))
url = cutoutbaseurl + '?' + query_string

urllib.request.urlretrieve(url, 'HCG7_SDSS_cutout.jpg')

In [ ]:
urllib.parse.urlencode(dict(ra=hcg7_center.ra.deg,
                                     dec=hcg7_center.dec.deg,
                                     width=impix, height=impix,
                                     scale=imsize.to(u.arcsec).value/impix))

Ahora miremos la imagen.

In [ ]:
IPython.display.Image('HCG7_SDSS_cutout.jpg')

¡Bonito!

## Transformando entre sistemas de coordenadas y planeación de observaciones

Supongamos que usted quiere estudiar alguno de los objetos en este catálogo, y usted quiere saber cuando puede observarlo. Afortunadamente `astropy.coordinates` nos da esta información.

### Introduciendo transformaciones de coordenadas

Para enteder lo que viene a continuación, recomendamos leer el [resumen del esquema de coordenadas de astropy](http://astropy.readthedocs.org/en/latest/coordinates/index.html#overview-of-astropy-coordinates-concepts).  Lo importante es saber que todas las coordenadas en astropy están en "marcos de referencia" particulares, y podemos transformar el mismo objeto `SkyCoord` de un marco a otro.  Pdemos transformar nuestro centro de HCG7 de ICRS a coordenadas galácticas:

In [ ]:
hcg7_center.galactic

In [ ]:
hcg7_center

Lo anterior nos permite hacer una visualización rápida del cambio de coordenadas. También es posible usar el método `transform_to()` para hacer este cambio.

In [ ]:
from astropy.coordinates import Galactic
hcg7_center.transform_to(Galactic())

In [ ]:
hcg7_center.galactic.ra  # esto debe dar un mensaje de error

In [ ]:
hcg7_center.galactic.b

### Cambio a coordenadas AltAz

Para hacer observaciones, necesitamos hacer un cambio de coordenadas a un sistema basado en donde está el observador. Lo más común es usar coordenadas horizontales, o "AltAz". Necesitamos especificar desde donde queremos observar.

In [ ]:
from astropy.coordinates import EarthLocation
from astropy.time import Time

observing_location = EarthLocation(lat='4d', lon='-72d', height=2600*u.m)  # Bogotá
# Si se usa astropy v1.1 o más reciente, se puede reemplazar por la ubicación del observatorio:
#observing_location = EarthLocation.of_site('Kitt Peak')

observing_time = Time('2015-12-21 1:00')
#observing_time = Time.now()

Ahora usamos lo anterior para crear un objeto en el marco `AltAz`. Este marco tiene información sobre la atmósfera, que puede ser usado para corregir por refracción atmosférica.

In [ ]:
from astropy.coordinates import AltAz

aa = AltAz(location=observing_location, obstime=observing_time)
aa

Ahora transformamos nuestro `SkyCoord` en ICRS a `AltAz` para encontrar la ubicación en el cielo.

In [ ]:
hcg7_center.transform_to(aa)

Ejercicio: Determine la longitud galáctica en ángulo horario a la que apunta el cenit en este momento en Medellín.

**Masa de aire:** Determinar *airmass* para programar observaciones.

In [ ]:
from astropy.coordinates import AltAz, EarthLocation, SkyCoord
from astropy.time import Time
import astropy.units as u

# 1. Definir ubicación y tiempo
observatorio = EarthLocation(lat=-30.16*u.deg, lon=-70.80*u.deg, height=2200*u.m) # CTIO
tiempo = Time('2026-04-27 02:00:00')

# 2. Definir el objeto y transformar a AltAz
target = SkyCoord.from_name('Sirius')
frame_altaz = AltAz(obstime=tiempo, location=observatorio)
sirius_altaz = target.transform_to(frame_altaz)

# 3. Obtener el Airmass
print(f"Altitud: {sirius_altaz.alt:.2f}")
print(f"Airmass: {sirius_altaz.secz:.2f}")

# Usando `coordinates` y `table` para combinar y comparar catálogos

Vamos a hacer un análisis multi-longitud de onda para caracterizar los objetos del grupo de galaxias HCG 7 (Hickson Compact Group 7).

Vamos a cruzar los datos en el óptico de SDSS con los de IR de 2MASS para:

### 1. Extender el Espectro de Observación (Óptico + IR)
* **SDSS (Sloan Digital Sky Survey):** Principalmente en el espectro visible. Rastrea luz de las estrellas jóvenes (en banda corta $g$ vs. banda larga $r$) y la estructura gaseosa.
* **2MASS (Two Micron All Sky Survey):** Datos en el infrarrojo cercano (bandas J, H, K). Permite ver a través del polvo interestelar y observar la luz de estrellas viejas y evolucionadas, que representan la mayor parte de la masa estelar de una galaxia.

### 2. Identificar Poblaciones Estelares
Al combinar ambos catálogos, se pueden calcular colores que mezclan ambas bandas (por ejemplo, entre una magnitud óptica de SDSS y una infrarroja de 2MASS). Esto ayuda a distinguir entre:
* **Galaxias con formación estelar activa:** (más brillantes en el azul/óptico).
* **Galaxias "pasivas" o elípticas:** (dominadas por estrellas rojas, muy brillantes en el infrarrojo).

### 3. Censo de Masa Estelar
El objetivo de buscar la contraparte en 2MASS de las galaxias detectadas por SDSS es obtener una estimación más precisa de la masa total de las galaxias del grupo HCG 7, ya que la emisión en el infrarrojo cercano está muy correlacionada con la masa estelar total.

### 4. Limpieza del campo (Estrellas vs Galaxias)
En campos densos, es fácil confundir una estrella de nuestra propia galaxia con una galaxia lejana. El cross-matching permite verificar si un objeto detectado como "borroso" en el óptico tiene las características de una fuente puntual o extendida en el infrarrojo, ayudando a confirmar que los miembros identificados realmente pertenecen al grupo galáctico estudiado.

### 5. Validación Estadística
Se calcula la distancia entre los vecinos más cercanos y se compara con una distribución aleatoria para demostrar que las coincidencias entre SDSS y 2MASS no son espurias, sino que estamos observando físicamente el mismo objeto en dos longitudes de onda distintas.

In [ ]:
import urllib
import IPython.display

In [ ]:
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.table import Table

In [ ]:
%matplotlib inline
from matplotlib import pyplot as plt
import numpy as np

Al final de la sección anterior, encontramos que HCG7 está en SDSS, lo que significa que podemos descargar catálogos de objetos directamente desde SDSS. Más adelante, combinaremos este catálogo con otro catálogo que cubra el mismo campo, permitiéndonos hacer gráficas combinando ambos catálogos.

In [ ]:
hcg7_center = SkyCoord.from_name('Hcg 7')
hcg7_center

Vamos a entrar a la base de datos SQL de SDSS usando el paquete [astroquery](https://astroquery.readthedocs.org).  This will require an internet connection and a working install of astroquery. Si usted no tiene este paquete, puede ignorar las próximas dos celdas, porque los archivos de datos están en el repositorio. Dependiendo de su versión de `astroquery`, puede que aparezca un Warning, que podemos ignorar sin problema.

En esta URL encontramos algunas opciones que se pueden poner en el query
http://skyserver.sdss.org/dr14/en/help/browser/browser.aspx#&&history=description+PhotoObj+V

In [ ]:
from astroquery.sdss import SDSS
sdss = SDSS.query_region(coordinates=hcg7_center, width=40*u.arcmin,height=40*u.arcmin,
                         spectro=True, #
                         photoobj_fields=['ra','dec','u','g','r','i','z'])

Los *queries* hechos a `astroquery` nos devuelve un objeto [`astropy.table.Table`](http://docs.astropy.org/en/stable/table/index.html).  Podemos trabajar con este objeto sin guardar nada en disco, pero en esta ocasión si lo haremos. De esta manera, al cerrar la sesión y volver, usted no tiene que hacer un *query* de nuevo.

(Esto no funcionará si usted no corrió la celda anterior. No hay problema, simplemente vaya a la celda que tiene ``Table.read`` y use la copia de esta tabla, incluida en el repositorio.)

In [ ]:
sdss.write('HCG7_SDSS_photo.dat', format='ascii',overwrite=True)

Si usted no tiene internet, simplemente puede leer la tabla desde Python corriendo la siguiente celda. Si usted hizo el query anterior, usted puede ignorar esta instrucción, dado que la tabla ya está almacenada en memoria como la variable llamada `sdss`.

In [ ]:
sdss = Table.read('HCG7_SDSS_photo.dat', format='ascii')

Ahora tenemos un catálogo de objetos que obtuvimos de SDSS. Supongamos que usted tiene su propio catálogo de objetos en el mismo campo para compararlo con el de SDSS. En este caso usaremos un catálogo extraído de [2MASS](http://www.ipac.caltech.edu/2mass/).  Vamos a cargar este catálogo (que está en el repositorio) a Python.

In [ ]:
!wget https://github.com/saint-germain/astropy/raw/refs/heads/master/HCG7_2MASS.tbl

In [ ]:
# Correr en caso de que el query as SDSS falle
!wget https://github.com/saint-germain/astropy/raw/refs/heads/master/HCG7_SDSS_photo.dat

In [ ]:
twomass = Table.read('HCG7_2MASS.tbl', format='ascii')

In [ ]:
sdss = Table.read('HCG7_SDSS_photo.dat', format='ascii')

Para combinar los catálogos, necesitamos objetos `SkyCoord`. Vamos a contruirlos a partir de las tablas que cargamos. Esto resulta ser directo: tomamos las columnas `ra` y `dec` de la tabla y se las damos al constructor `SkyCoord` constructor.  Primero hagamos una inspección de las tablas.

In [ ]:
sdss[0:5]

In [ ]:
twomass[0:5]

Ya que tenemos las columnas de `ra` y `dec` podemos usarlas para crear nuestros `SkyCoord`s.

No es necesario crear un `SkyCoord` para cada fila en la tabla. En vez de esto, aprovechamos que `SkyCoord` recibe *arrays* de valores de coordenadas, ya sean `Quantity`s, listas de *strings*, columnas de `Table`s, etc., y `SkyCoord` hará las operaciones tranquilamente elemento a elemento.

In [ ]:
coo_sdss = SkyCoord(sdss['ra']*u.deg, sdss['dec']*u.deg)
coo_twomass = SkyCoord(twomass['ra'], twomass['dec'])

In [ ]:
coo_twomass

Hay una diferencia sutil. Para SDSS tuvimos que dar unidades, pero no para 2MASS. Esto es porque la tabla de 2MASS tiene unidades asociadas a las columnas, mientras que SDSS no tiene unidades.

Ahora usamos el método ``SkyCoord.match_to_catalog_sky`` para combinar ambos catálogos. El orden importa: combinamos 2MASS sobre SDSS porque hay muchas más entradas en SDSS, de manera que es probable que casi todos los objetos de 2MASS están en SDSS y no al revés.

In [ ]:
idx_sdss, d2d_sdss, d3d_sdss = coo_twomass.match_to_catalog_sky(coo_sdss)

``idx`` da los índices de ``coo_sdss`` que se ajustan mejor a SDSS, mientras ``d2d`` y ``d3d`` son las distancias en cielo y en espacio real entre posibles coincidencias. En nuestro caso ignoraremos ``d3d`` porque no dimos información sobre la distancia a lo largo de la línea de visión. En cambio ``d2d`` nos da un buen diagnóstico de las posibles coincidencias:

In [ ]:
plt.hist(d2d_sdss.arcsec, histtype='step', range=(0,2))
plt.xlabel('Separation [arcsec]')
plt.tight_layout()

Ok, todos están dentro de un arcosegundo, lo cual es prometedor. Pero, ¿estamos seguros de que no es sólo que cualquier objeto tendría coincidencias dentro de un segundo de arco? Vamos a comprobarlo comparando con un conjunto de puntos aleatorios.

Primero creamos un conjunto de puntos uniformemente aleatorios (con un tamaño que coincida con `coo_twomass`) que cubran el mismo rango de RA/Decs que hay en `coo_sdss`. Esto lo haremos con el método `ptp()` (peak-to-peak), que es equivalente a tomar la diferencia entre el valor máximo y mínimo de un arreglo.

In [ ]:
ras_sim = (np.random.rand(len(coo_twomass)) * (coo_sdss.ra.max() - coo_sdss.ra.min())) + coo_sdss.ra.min()
decs_sim = (np.random.rand(len(coo_twomass)) * (coo_sdss.dec.max() - coo_sdss.dec.min())) + coo_sdss.dec.min()
ras_sim, decs_sim

Observemos que no es necesario especificar explícitamente las unidades para `ras_sim` y `decs_sim`, porque ya son objetos con unidad `Angle` porque fueron creados desde `coo_sdss.ra`y `coo_sdss.dec`.

Ahora creamos un objeto `SkyCoord` a partir de estos puntos y lo cotejamos con `coo_sdss` tal y como hicimos anteriormente para 2MASS.

In [ ]:
coo_simulated = SkyCoord(ras_sim, decs_sim)
idx_sim, d2d_sim, d3d_sim = coo_simulated.match_to_catalog_sky(coo_sdss)

In [ ]:
plt.hist(d2d_sim.arcsec, bins='auto', histtype='step', label='Simulated', linestyle='dashed')
plt.hist(d2d_sdss.arcsec, bins='auto', histtype='step', label='2MASS')
plt.xlabel('separation [arcsec]')
plt.legend(loc=0)
plt.tight_layout()

Muy bien, parece que las fuentes colocadas al azar están a un minuto de arco de distancia, así que probablemente podemos confiar en que nuestras primeras coincidencias (que estaban a un segundo de arco) son válidas.

Ahora podemos calcular cosas como colores que combinen la fotometría de SDSS y 2MASS.

In [ ]:
rmag = sdss['r'][idx_sdss]
grcolor = sdss['g'][idx_sdss] - rmag
rKcolor = rmag - twomass['k_m_ext']

plt.subplot(1, 2, 1)
plt.scatter(rKcolor, rmag)
plt.xlabel('r-K')
plt.ylabel('r')
plt.xlim(2.5, 4)
plt.ylim(18, 12) #mags go backwards!

plt.subplot(1, 2, 2)
plt.scatter(rKcolor, rmag)
plt.xlabel('r-K')
plt.ylabel('g-r')
plt.xlim(2.5, 4)
plt.tight_layout()

## 1.  $r$ vs. $r-K$ (Diagrama color-magnitud)

Relación entre la masa estelar y la historia de formación estelar.

- Los puntos en la parte superior izquierda (más brillantes en $r$, más azules en $r-K$) representan espirales grandes como HCG 7a y 7c. Aún tienen suficiente gas para mantener poblaciones estelares más “azules” (jóvenes) y aportan la mayor cantidad de masa al grupo.  
- El grupo de puntos entre $r = 16$ y $r = 18$ que son bastante rojos ($r-K \approx 3.3$ a $3.8$) probablemente corresponde a los miembros más pequeños y a aquellos ya “apagados” (quenched), como HCG 7b y 7d.

## 2. $g-r$ vs. $r-K$ (Diagrama color-color)


- A medida que las galaxias se vuelven más rojas en el óptico ($g-r$), también lo hacen en óptico–IR ($r-K$). Esto implica que ambos colores trazan las mismas tendencias
- Hay un “salto”: escasez relativa de puntos en la parte media del eje $g-r$ (alrededor de $15.5$ a $16.5$). Este es el famoso "valle verde". Una vez que una galaxia empieza a perder su gas debido a interacciones de marea con sus vecinas, no permanece “verde” por mucho tiempo: se mueve rápidamente hacia la secuencia roja, donde experimenta *quenching* (el grupo en la parte superior derecha).  
- Como los puntos formen una secuencia estrecha, la metalicidad entre estas galaxias es relativamente similar, y el principal factor que impulsa las diferencias de color es la edad de la población estelar junto con el cese abrupto de la formación estelar.

## Diagnóstico de HCG 7

Si este fuera un grupo “joven” de galaxias que apenas se están ensamblando por primera vez, se verían muchos más puntos en la esquina inferior izquierda del diagrama color-magnitud (galaxias con formación estelar activa pero pocas estrellas viejas).

La “cola” de galaxias rojas y débiles indica que el entorno está despojando eficientemente el gas de los miembros más pequeños, transformándolos en galaxias S0 pasivas, mientras que las espirales más grandes probablemente sean las siguientes candidatas para este proceso de *quenching* a medida que continúan interactuando.


## Resumen

Si solo se usa SDSS, se puede confundir una galaxia espiral polvorienta (joven) con una galaxia elíptica vieja. Si solo se usa 2MASS, no sería posible distinguir si una galaxia está formando estrellas activamente o si simplemente es muy masiva. Al combinarlos en el diagrama $(g-r)$ vs. $(r-K)$ se determina cuánto gas le queda a las galaxias de HCG 7 para seguir formando estrellas.

## 1. Qué aporta SDSS ($g, r$)


- **Sensibilidad a estrellas jóvenes:** Las bandas $g$ ("azul") y $r$ ("roja") están dominadas por la luz de estrellas jóvenes y masivas. Si una galaxia en HCG 7 tiene incluso una pequeña cantidad de formación estelar reciente (como los brazos espirales en NGC 192), aparecerá en las bandas de SDSS.  
- **Sensibilidad al polvo:** Observaciones en el óptico son susceptibles a la extinción por polvo. El color $g-r$ puede parecer más “rojo” no porque las estrellas sean viejas, sino porque el polvo está bloqueando la luz azul (extinción selectiva).  
- **Morfología:** SDSS tiene una resolución angular mucho mayor que 2MASS, lo que permite ver los brazos espirales y las distorsiones de marea que indican el estado de interacción del grupo.

## 2. Qué aporta 2MASS ($K$)

- **Trazador de masa estelar:** La banda $K$ ($2.2\,\mu\text{m}$) es relativamente poco afectada por el polvo y por la presencia de estrellas jóvenes (azules) de vida corta. Traza las estrellas viejas y de baja masa que constituyen la mayor parte de la masa de una galaxia.  
- **El índice $r-K$:** Al restar la magnitud $K$ de 2MASS a la magnitud $r$ de SDSS, se compara la masa total (K) con la población estelar de edad intermedia (r).  
- **Enfriamento (quenching):** Si una galaxia es brillante en 2MASS pero débil en SDSS, se sabe que es una galaxia masiva que ha dejado completamente de formar estrellas (una elíptica o S0 “roja y muerta”).



# Ejercicio

Encuentre los horarios de cada día durante los cuales el Centro Galáctico es visible desde Medellín durante los próximos dos meses.

# Ejercicio

Usando la función `astropy.coordinates.get_sun` junto con la clase `astropy.time.Time` haga un analema completo del Sol a las 3 pm visto desde Medellín.

# Ejercicio

Usando la función `astropy.coordinates.get_constellation` y construyendo una malla de coordenadas galácticas con $\Delta l,\Delta b=2$ deg, genere una lista con las constelaciones que quedan a menos de 10 grados de latitud galáctica del plano galáctico.

# Ejercicio

Calcule el valor del coeficiente de Einstein A para la transición rotacional 1 a 0 del CO (dipolo eléctrico $\mu=0.112$ Debye, constante rotacional $B=1.922$ cm^-1 - número de onda) y para el CS ($\mu=1.96$ Debye, $B=0.817$ cm^-1) usando la siguiente fórmula (use $J=0$):

$$A = \frac{64\pi^4}{3hc^3} \mu^2 \frac{J + 1}{2J + 3} \nu^3;\quad \nu=B(J+1)$$

Ayuda: También se puede usar la conversión 1 Debye = 1e-18 cm^5/2 g^1/2 s^-1

Compare con el resultado de un query a la Leiden Atomic and Molecular Database, aprovechando el ejemplo siguiente:


In [ ]:
from astroquery.lamda import Lamda

collrates, radtransitions, enlevels = Lamda.query(mol='co')
j1_0 = radtransitions[radtransitions['Upper'] == 2]  # Index 2 is J=1
print(j1_0['EinsteinA'])